In [49]:
"""
Publication-quality figures from IBM Quantum hardware results
ibm_torino · 10,000 shots/basis · Implementation A (Lloyd-type CTC emulation)

Figures:
  fig1_psucc.pdf         — p_succ: hardware vs ideal, Clopper–Pearson CI
  fig2_pauli.pdf         — Pauli expectations: hardware vs ideal
  fig3_bloch.pdf         — Bloch sphere: ρ_M (ideal) vs ρ_Y (hardware)
  fig4_bootstrap.pdf     — Bootstrap F distribution with CI bands
  fig5_densitymatrix.pdf — Re/Im density matrix: hardware vs ideal
  fig6_raw_counts.pdf    — Raw measurement histogram (all 8 outcomes, 3 bases)
  fig_panel.pdf          — Combined panel for paper
"""

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d

# ── Load hardware data ────────────────────────────────────────────────────────
with open('/Users/nandan/Desktop/CTCs/IBM/hardware_postselection_results.json') as f:
    D = json.load(f)

# ── Ideal reference values (from ρ_M) ────────────────────────────────────────
sx_ideal  = -0.2384;  sy_ideal = 0.5179;  sz_ideal = -0.8215
rho_M_re  = [[0.08923, -0.11921], [-0.11921,  0.91077]]
rho_M_im  = [[0.0,    -0.25895], [ 0.25895,  0.0    ]]

# ── Unpack hardware values ────────────────────────────────────────────────────
p_hw      = D['p_succ']['estimate']
p_lo      = D['p_succ']['ci_lo']
p_hi      = D['p_succ']['ci_hi']

sx_hw     = D['bloch_vector']['sx']
sy_hw     = D['bloch_vector']['sy']
sz_hw     = D['bloch_vector']['sz']

F_hw      = D['fidelity']['point_estimate']
F_lo      = D['fidelity']['ci_lo']
F_hi      = D['fidelity']['ci_hi']
F_std     = D['fidelity']['bootstrap_std']
F_med     = D['fidelity']['bootstrap_median']
bsF       = np.array(D['bootstrap_F_samples'])

rho_hw_re = [[D['rho_Y'][i][j]['re'] for j in range(2)] for i in range(2)]
rho_hw_im = [[D['rho_Y'][i][j]['im'] for j in range(2)] for i in range(2)]

shots     = D['metadata']['shots_per_basis']
n_kept_Z  = sum(D['postselected']['Z'].values())
raw       = D['raw_counts']

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'       : 'DejaVu Serif',
    'font.size'         : 11,
    'axes.titlesize'    : 12,
    'axes.labelsize'    : 11,
    'xtick.labelsize'   : 10,
    'ytick.labelsize'   : 10,
    'legend.fontsize'   : 9.5,
    'figure.dpi'        : 180,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.linewidth'    : 0.8,
    'xtick.direction'   : 'in',
    'ytick.direction'   : 'in',
    'pdf.fonttype'      : 42,   # embeds fonts for journal submission
})

C_HW    = '#C0392B'   # deep red   — hardware
C_IDEAL = '#27AE60'   # green      — ideal/theory
C_SHADE = '#FADBD8'   # light red shading

OUT = '/Users/nandan/Desktop/CTCs/IBM/Figures_PCTC_Hard'

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 1 — p_succ bar chart with CP CI
# ═══════════════════════════════════════════════════════════════════════════════
def fig_psucc():
    fig, ax = plt.subplots(figsize=(3.8, 3.5))

    x      = [0, 1]
    vals   = [0.25,  p_hw]
    colors = [C_IDEAL, C_HW]
    elo    = [0,      0]
    ehi    = [0,      min(p_hi, 0.25) - p_hw]
    labels = ['Ideal\n(theory)', f'Hardware\n(ibm_torino)']

    bars = ax.bar(x, vals, color=colors, width=0.5,
                  yerr=[elo, ehi],
                  error_kw=dict(elinewidth=1.6, capsize=6, capthick=1.6,
                                ecolor='#222222'),
                  zorder=3, edgecolor='white', linewidth=0.6)
    bars[0].set_hatch('///')

    ax.axhline(0.25, color=C_IDEAL, ls='--', lw=1.2, alpha=0.6, zorder=2)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10.5)
    ax.set_ylim(0, 0.32)
    ax.set_ylabel('Post-selection probability $p_{\\mathrm{succ}}$')
    ax.set_title('(a) Post-selection rate', pad=8)
    ax.yaxis.grid(True, alpha=0.3, zorder=0)

    for xi, val in zip(x, vals):
        ax.text(xi, val + 0.014, f'{val:.4f}', ha='center',
                fontsize=9, color='#111111')

    ci_txt = f'95% CI  [{p_lo:.4f}, {p_hi:.4f}]\n$N_{{\\rm shots}}={shots:,}$  /  basis'
    ax.text(0.97, 0.97, ci_txt, transform=ax.transAxes,
            ha='right', va='top', fontsize=8.5, color='#444',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc', lw=0.7))

    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 2 — Pauli expectations
# ═══════════════════════════════════════════════════════════════════════════════
def fig_pauli():
    fig, ax = plt.subplots(figsize=(4.8, 3.6))

    keys   = ['X', 'Y', 'Z']
    xlbls  = ['$\\langle X\\rangle$', '$\\langle Y\\rangle$', '$\\langle Z\\rangle$']
    ideal  = [sx_ideal, sy_ideal, sz_ideal]
    hw     = [sx_hw,    sy_hw,    sz_hw   ]

    x = np.arange(3); w = 0.3
    b1 = ax.bar(x - w/2, ideal, width=w, color=C_IDEAL, label='Ideal ($\\rho_M$)',
                edgecolor='white', zorder=3, hatch='///')
    b2 = ax.bar(x + w/2, hw,    width=w, color=C_HW,    label='Hardware (ibm_torino)',
                edgecolor='white', zorder=3)

    ax.axhline(0, color='#555', lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(xlbls, fontsize=13)
    ax.set_ylim(-1.15, 1.15)
    ax.set_ylabel('Expectation value')
    ax.set_title('(b) Pauli expectation values', pad=8)
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    ax.legend(framealpha=0.9, edgecolor='#ccc')

    for xi, (iv, hv) in enumerate(zip(ideal, hw)):
        ax.text(xi - w/2, iv + (0.05 if iv >= 0 else -0.10),
                f'{iv:.3f}', ha='center', fontsize=8.5, color=C_IDEAL)
        ax.text(xi + w/2, hv + (0.05 if hv >= 0 else -0.10),
                f'{hv:.3f}', ha='center', fontsize=8.5, color=C_HW)

    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 3 — Bloch sphere
# ═══════════════════════════════════════════════════════════════════════════════
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0,0),(0,0), *args, **kwargs)
        self._verts3d = xs, ys, zs
    def do_3d_projection(self, renderer=None):
        xs,ys,zs = proj3d.proj_transform(*self._verts3d, self.axes.M)
        self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))
        return np.min(zs)

def draw_bloch(ax, vectors, labels, colors):
    u = np.linspace(0, 2*np.pi, 60)
    v = np.linspace(0, np.pi, 40)
    ax.plot_wireframe(np.outer(np.cos(u),np.sin(v)),
                      np.outer(np.sin(u),np.sin(v)),
                      np.outer(np.ones(60),np.cos(v)),
                      rstride=5, cstride=5, color='#cccccc', lw=0.3, alpha=0.35)
    for xyz, lbl in [([1.25,0,0],'$x$'),([0,1.25,0],'$y$'),([0,0,1.25],'$z$'),
                     ([-1.25,0,0],''),([ 0,-1.25,0],''),([ 0,0,-1.25],'')]:
        ax.plot([0,xyz[0]],[0,xyz[1]],[0,xyz[2]],
                color='#999999', lw=0.7, alpha=0.7)
        if lbl: ax.text(xyz[0]*1.12,xyz[1]*1.12,xyz[2]*1.12,
                        lbl, fontsize=10, ha='center', color='#555')
    t = np.linspace(0, 2*np.pi, 120)
    for c1,c2,c3 in [(np.cos(t),np.sin(t),np.zeros(120)),
                     (np.cos(t),np.zeros(120),np.sin(t)),
                     (np.zeros(120),np.cos(t),np.sin(t))]:
        ax.plot(c1,c2,c3, color='#bbbbbb', lw=0.5, alpha=0.45)

    for (bx,by,bz), lbl, col in zip(vectors, labels, colors):
        arw = Arrow3D([0,bx],[0,by],[0,bz],
                      mutation_scale=16, lw=2.4, arrowstyle='-|>', color=col)
        ax.add_artist(arw)
        ax.text(bx*1.15, by*1.15, bz*1.15, lbl,
                fontsize=10, color=col, fontweight='bold')

    ax.set_xlim(-1.35,1.35); ax.set_ylim(-1.35,1.35); ax.set_zlim(-1.35,1.35)
    ax.set_box_aspect([1,1,1]); ax.axis('off')

def fig_bloch():
    fig = plt.figure(figsize=(5.0, 4.6))
    ax  = fig.add_subplot(111, projection='3d')

    draw_bloch(ax,
               vectors=[[sx_ideal, sy_ideal, sz_ideal],
                         [sx_hw,    sy_hw,    sz_hw   ]],
               labels =['$\\rho_M$', '$\\rho_Y^{\\rm hw}$'],
               colors =[C_IDEAL,     C_HW              ])

    patches = [mpatches.Patch(color=C_IDEAL, label='$\\rho_M$ (ideal)'),
               mpatches.Patch(color=C_HW,    label='$\\rho_Y^{\\rm hw}$ (ibm_torino)')]
    ax.legend(handles=patches, loc='upper left',
              bbox_to_anchor=(-0.08, 0.98), framealpha=0.9, edgecolor='#ccc')

    # Annotate Bloch vector magnitudes
    r_ideal = (sx_ideal**2+sy_ideal**2+sz_ideal**2)**0.5
    r_hw    = (sx_hw**2   +sy_hw**2   +sz_hw**2   )**0.5
    ax.text2D(0.02, 0.04,
              f'$|\\mathbf{{r}}_M|={r_ideal:.3f}$ (pure)\n'
              f'$|\\mathbf{{r}}_Y|={r_hw:.3f}$ (mixed by noise)',
              transform=ax.transAxes, fontsize=9, color='#333',
              bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc', lw=0.6))

    ax.set_title('(c) Bloch sphere', fontsize=12, pad=4)
    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 4 — Bootstrap F histogram
# ═══════════════════════════════════════════════════════════════════════════════
def fig_bootstrap():
    fig, ax = plt.subplots(figsize=(5.2, 3.6))

    n, bins, patches_hist = ax.hist(
        bsF, bins=50, color=C_HW, alpha=0.80,
        edgecolor='white', linewidth=0.4, zorder=3,
        label=f'Bootstrap resamples ($N={len(bsF):,}$)')

    # CI shading
    ax.axvspan(F_lo, F_hi, alpha=0.18, color=C_HW, zorder=1)

    # Key lines
    ax.axvline(F_med, color=C_HW,    lw=2.0, zorder=4,
               label=f'Median $F={F_med:.4f}$')
    ax.axvline(F_lo,  color=C_HW,    lw=1.2, ls='--', zorder=4)
    ax.axvline(F_hi,  color=C_HW,    lw=1.2, ls='--', zorder=4,
               label=f'95% CI  [{F_lo:.4f}, {F_hi:.4f}]')
    ax.axvline(1.0,   color=C_IDEAL, lw=1.6, ls=':', zorder=4,
               label='Ideal $F=1$')

    # Annotate CI bracket
    ymax = ax.get_ylim()[1]
    # ax.annotate('', xy=(F_hi, ymax*0.88), xytext=(F_lo, ymax*0.88),
    #             arrowprops=dict(arrowstyle='<->', color=C_HW, lw=1.3))
    
    ax.text((F_lo+F_hi)/2, ymax*0.91, f'',
            ha='center', fontsize=8.5, color=C_HW, zorder=5)

    ax.set_xlabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_ylabel('Bootstrap count')
    ax.set_title(f'(d) Bootstrap distribution of $F(\\rho_Y,\\rho_M)$\n'
                 f'ibm_torino  ·  $N_{{\\rm kept}}\\approx{n_kept_Z:,}$ post-selected shots',
                 pad=6)
    ax.legend(framealpha=0.9, edgecolor='#ccc', loc='upper right', bbox_to_anchor=(0.95, 1.0))
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 5 — Density matrix Re / Im side by side
# ═══════════════════════════════════════════════════════════════════════════════
def fig_densitymatrix():
    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.8))

    elems  = [(0,0),(0,1),(1,0),(1,1)]
    xlbls  = ['$|00\\rangle$','$|01\\rangle$','$|10\\rangle$','$|11\\rangle$']
    x      = np.arange(4); w = 0.3

    for ax, part, title, rho_ideal_arr, rho_hw_arr in zip(
            axes,
            ['re', 'im'],
            ['(e) $\\mathrm{Re}(\\rho_Y)$', '(f) $\\mathrm{Im}(\\rho_Y)$'],
            [rho_M_re, rho_M_im],
            [rho_hw_re, rho_hw_im]):

        ideal_vals = [rho_ideal_arr[i][j] for i,j in elems]
        hw_vals    = [rho_hw_arr[i][j]    for i,j in elems]

        b1 = ax.bar(x - w/2, ideal_vals, width=w, color=C_IDEAL,
                    label='Ideal ($\\rho_M$)', edgecolor='white', zorder=3, hatch='///')
        b2 = ax.bar(x + w/2, hw_vals,    width=w, color=C_HW,
                    label='Hardware', edgecolor='white', zorder=3)

        ax.axhline(0, color='#555', lw=0.8)
        ax.set_xticks(x); ax.set_xticklabels(xlbls, fontsize=9.5)
        ax.set_title(title, fontsize=12)
        ax.set_ylabel('Matrix element value')
        ax.yaxis.grid(True, alpha=0.3, zorder=0)
        ax.set_ylim(-0.70, 1.08)
        ax.legend(framealpha=0.9, edgecolor='#ccc', fontsize=9)

        # Value annotations on hardware bars
        for xi, hv in zip(x, hw_vals):
            if abs(hv) > 0.005:
                offset = 0.04 if hv >= 0 else -0.09
                ax.text(xi + w/2 + 0.18, hv + offset, f'{hv:.3f}', ha='center', fontsize=8.5, color=C_HW,
                zorder=5)

    fig.suptitle('Reconstructed $\\rho_Y$ vs. ideal $\\rho_M$  '
                 '(ibm_torino, post-selected tomography)',
                 fontsize=11, y=1.02)
    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  FIG 6 — Raw measurement histogram (all 8 bitstrings, 3 bases)
# ═══════════════════════════════════════════════════════════════════════════════
def fig_raw_counts():
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4), sharey=False)

    # All 3-bit outcomes in fixed order
    all_bs = ['000','001','010','011','100','101','110','111']
    # Colour: green if crR=0 & crG=0 (post-selected), grey otherwise
    bar_colors = []
    for bs in all_bs:
        crT, crG, crR = bs[0], bs[1], bs[2]
        bar_colors.append('#27AE60' if (crR=='0' and crG=='0') else '#BDC3C7')

    basis_titles = {'Z': '(f) Z basis', 'X': '(g) X basis', 'Y': '(h) Y basis'}

    for ax, basis in zip(axes, ['Z','X','Y']):
        counts = raw[basis]
        vals   = [counts.get(bs, 0) for bs in all_bs]
        total  = sum(vals)
        probs  = [v/total for v in vals]

        bars = ax.bar(range(8), probs, color=bar_colors,
                      edgecolor='white', linewidth=0.5, zorder=3)

        # Highlight post-selected bars with edge
        for i, (bs, bar) in enumerate(zip(all_bs, bars)):
            crT, crG, crR = bs[0], bs[1], bs[2]
            if crR=='0' and crG=='0':
                bar.set_edgecolor('#1A6B3C')
                bar.set_linewidth(1.5)

        ax.set_xticks(range(8))
        ax.set_xticklabels(all_bs, fontsize=8.5, rotation=45, ha='right')
        ax.set_xlabel('Bitstring (crTomo|crG|crR)', fontsize=9)
        ax.set_ylabel('Probability' if basis=='Z' else '')
        ax.set_title(f'({["Z","X","Y"].index(basis)+6}) {basis}-basis  '
                     f'({total:,} shots)', fontsize=11)
        ax.yaxis.grid(True, alpha=0.3, zorder=0)

        # Mark ideal p_succ line (0.25/4 = 0.0625 per post-selected outcome)
        p_kept = sum(probs[i] for i,bs in enumerate(all_bs)
                     if bs[2]=='0' and bs[1]=='0')
        ax.axhline(p_kept, color='#C0392B', ls='--', lw=1.1, alpha=0.7,
                   label=f'$p_{{\\rm succ}}$={p_kept:.3f}')
        ax.legend(fontsize=8.5, framealpha=0.9, bbox_to_anchor=(1.0, 0.95), loc='upper right')

    # Legend for bar colours
    patch_kept   = mpatches.Patch(color='#27AE60', label='Post-selected (crG=crR=0)')
    patch_disc   = mpatches.Patch(color='#BDC3C7', label='Discarded')
    fig.legend(handles=[patch_kept, patch_disc],
           loc='lower center', ncol=2, fontsize=9.5,
           bbox_to_anchor=(0.5, -0.07), framealpha=0.9, edgecolor='#ccc')

    fig.suptitle('Raw measurement histograms — ibm_torino  '
                 '(green bars = post-selected Bell outcome $|\\Phi^+\\rangle$)',
                 fontsize=11, y=1.02)
    fig.tight_layout()
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  COMBINED PANEL  — 2×3 for paper
# ═══════════════════════════════════════════════════════════════════════════════
def fig_panel():
    fig = plt.figure(figsize=(14.5, 9.0))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.46, wspace=0.36)

    # ── (a) p_succ ─────────────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0,0])
    x=[0,1]; vals=[0.25,p_hw]; colors=[C_IDEAL,C_HW]
    elo    = [0,      0]
    ehi    = [0,      min(p_hi, 0.25) - p_hw]
    bars=ax1.bar(x,vals,color=colors,width=0.5,
                 yerr=[elo,ehi],
                 error_kw=dict(elinewidth=1.4,capsize=6,capthick=1.4,ecolor='#222'),
                 zorder=3,edgecolor='white')
    bars[0].set_hatch('///')
    ax1.axhline(0.25,color=C_IDEAL,ls='--',lw=1.1,alpha=0.6)
    ax1.set_xticks(x); ax1.set_xticklabels(['Ideal','Hardware'],fontsize=10)
    ax1.set_ylim(0,0.31); ax1.set_ylabel('$p_{\\rm succ}$')
    ax1.set_title('(a) Post-selection rate',fontsize=11)
    ax1.yaxis.grid(True,alpha=0.3,zorder=0)
    for xi,val in zip(x,vals):
        ax1.text(xi, val+0.014, f'{val:.4f}', ha='center', fontsize=9)

    # ── (b) Pauli ──────────────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0,1])
    keys=['X','Y','Z']; xlbls2=['$\\langle X\\rangle$','$\\langle Y\\rangle$','$\\langle Z\\rangle$']
    ideal_p=[sx_ideal,sy_ideal,sz_ideal]; hw_p=[sx_hw,sy_hw,sz_hw]
    x2=np.arange(3); w2=0.3
    ax2.bar(x2-w2/2,ideal_p,width=w2,color=C_IDEAL,label='Ideal',edgecolor='white',zorder=3,hatch='///')
    ax2.bar(x2+w2/2,hw_p,   width=w2,color=C_HW,   label='Hardware',edgecolor='white',zorder=3)
    ax2.axhline(0,color='#555',lw=0.8)
    ax2.set_xticks(x2); ax2.set_xticklabels(xlbls2,fontsize=12)
    ax2.set_ylim(-1.15,1.15); ax2.set_ylabel('Expectation value')
    ax2.set_title('(b) Pauli expectations',fontsize=11)
    ax2.yaxis.grid(True,alpha=0.3,zorder=0); ax2.legend(fontsize=9,framealpha=0.9)
    for xi,(iv,hv) in enumerate(zip(ideal_p,hw_p)):
        ax2.text(xi-w2/2,iv+(0.06 if iv>=0 else -0.12),f'{iv:.3f}',ha='center',fontsize=8,color=C_IDEAL)
        ax2.text(xi+w2/2,hv+(0.06 if hv>=0 else -0.12),f'{hv:.3f}',ha='center',fontsize=8,color=C_HW)

    # ── (c) Bootstrap ──────────────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0,2])
    ax3.hist(bsF,bins=50,color=C_HW,alpha=0.80,edgecolor='white',lw=0.4,zorder=3)
    ax3.axvspan(F_lo,F_hi,alpha=0.18,color=C_HW,zorder=1)
    ax3.axvline(F_med,color=C_HW,   lw=2.0,zorder=4,label=f'Median={F_med:.4f}')
    ax3.axvline(F_lo, color=C_HW,   lw=1.2,ls='--',zorder=4)
    ax3.axvline(F_hi, color=C_HW,   lw=1.2,ls='--',zorder=4,
                label=f'95% CI [{F_lo:.4f},{F_hi:.4f}]')
    ax3.axvline(1.0,  color=C_IDEAL,lw=1.5,ls=':',zorder=4,label='Ideal $F=1$')
    ax3.set_xlabel('$F(\\rho_Y,\\rho_M)$'); ax3.set_ylabel('Count')
    ax3.set_title('(c) Bootstrap $F$ distribution',fontsize=11)
    ax3.legend(fontsize=8.5,framealpha=0.9); ax3.yaxis.grid(True,alpha=0.3,zorder=0)

    # ── (d) Bloch sphere ───────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1,0], projection='3d')
    draw_bloch(ax4,
               [[sx_ideal,sy_ideal,sz_ideal],[sx_hw,sy_hw,sz_hw]],
               ['$\\rho_M$','$\\rho_Y$'],
               [C_IDEAL, C_HW])
    patches=[mpatches.Patch(color=C_IDEAL,label='$\\rho_M$ (ideal)'),
             mpatches.Patch(color=C_HW,   label='$\\rho_Y$ (hw)')]
    ax4.legend(handles=patches,loc='upper left',bbox_to_anchor=(-0.12,1.0),
               fontsize=8.5,framealpha=0.9)
    ax4.set_title('(d) Bloch sphere',fontsize=11,pad=2)

    # ── (e)/(f) Density matrix ─────────────────────────────────────────────────
    for col_idx, (part, title, rho_ideal_arr, rho_hw_arr) in enumerate([
            ('re','(e) $\\mathrm{Re}(\\rho_Y)$', rho_M_re, rho_hw_re),
            ('im','(f) $\\mathrm{Im}(\\rho_Y)$', rho_M_im, rho_hw_im)]):
        ax = fig.add_subplot(gs[1, 1+col_idx])
        elems=[(0,0),(0,1),(1,0),(1,1)]
        xlbls_dm=['$|00\\rangle$','$|01\\rangle$','$|10\\rangle$','$|11\\rangle$']
        iv=[rho_ideal_arr[i][j] for i,j in elems]
        hv=[rho_hw_arr[i][j]   for i,j in elems]
        xdm=np.arange(4); wdm=0.3
        ax.bar(xdm-wdm/2,iv,width=wdm,color=C_IDEAL,edgecolor='white',zorder=3,hatch='///',label='Ideal')
        ax.bar(xdm+wdm/2,hv,width=wdm,color=C_HW,   edgecolor='white',zorder=3,label='Hardware')
        ax.axhline(0,color='#555',lw=0.8)
        ax.set_xticks(xdm); ax.set_xticklabels(xlbls_dm,fontsize=8.5)
        ax.set_title(title,fontsize=11); ax.set_ylabel('Matrix element')
        ax.yaxis.grid(True,alpha=0.3,zorder=0); ax.set_ylim(-0.70,1.08)
        ax.legend(fontsize=8.5,framealpha=0.9)

    fig.suptitle(
        f'Implementation A — Post-selected YK Decoder on ibm_torino  '
        f'($N_{{\\rm shots}}={shots:,}$/basis,  '
        f'$F={F_hw:.4f}$, 95% CI $[{F_lo:.4f},\\,{F_hi:.4f}]$)',
        fontsize=11.5, y=1.005
    )
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  Save everything
# ═══════════════════════════════════════════════════════════════════════════════
print('Generating figures from ibm_torino hardware data ...')
figs = [
    ('fig1_psucc.png',         fig_psucc),
    ('fig2_pauli.png',         fig_pauli),
    ('fig3_bloch.png',         fig_bloch),
    ('fig4_bootstrap.png',     fig_bootstrap),
    ('fig5_densitymatrix.png', fig_densitymatrix),
    ('fig6_raw_counts.png',    fig_raw_counts),
    ('fig_panel.png',          fig_panel),
]

for fname, fn in figs:
    f = fn()
    f.savefig(OUT + fname, bbox_inches='tight', dpi=200)
    plt.close(f)
    print(f'  [✓] {fname}')

print(f'\nAll figures saved.')
print(f'\nKey hardware results:')
print(f'  Backend  : ibm_torino  ({shots:,} shots/basis)')
print(f'  p_succ   : {p_hw:.4f}  [{p_lo:.4f}, {p_hi:.4f}]  (ideal 0.2500)')
print(f'  F(ρY,ρM) : {F_hw:.4f}  [{F_lo:.4f}, {F_hi:.4f}]  (ideal 1.0000)')
print(f'  Purity   : {0.7563:.4f}  (ideal 1.0000)')
print(f'  |r| Bloch: {(sx_hw**2+sy_hw**2+sz_hw**2)**0.5:.4f}  (ideal 1.0000)')

Generating figures from ibm_torino hardware data ...
  [✓] fig1_psucc.png
  [✓] fig2_pauli.png
  [✓] fig3_bloch.png
  [✓] fig4_bootstrap.png
  [✓] fig5_densitymatrix.png
  [✓] fig6_raw_counts.png
  [✓] fig_panel.png

All figures saved.

Key hardware results:
  Backend  : ibm_torino  (10,000 shots/basis)
  p_succ   : 0.2430  [0.2346, 0.2515]  (ideal 0.2500)
  F(ρY,ρM) : 0.8577  [0.8411, 0.8749]  (ideal 1.0000)
  Purity   : 0.7563  (ideal 1.0000)
  |r| Bloch: 0.7159  (ideal 1.0000)
